# 03 · Pipeline completa e submission

1. Feature dal tracking → punteggi per frame per H/A/D (+ Smart Tagging come prior)
2. **HMM a durata minima** decodificato con Viterbi, per tempo: niente possessi di 3 frame, cambi di stato penalizzati
3. Scrittura del CSV, validazione, controllo di riproducibilità

Parametri: `solution/params.json` (dal notebook 02). Se manca, si usano i default scritti a mano.

In [ ]:
# --- Setup (locale o server Google) ---------------------------------------------
# Su Colab/Vertex: decommenta, clona la repo e indica dove hai messo i dati.
# !git clone <URL_REPO> mtah && %cd mtah
# !pip install -q -r solution/requirements.txt
# os.environ["MTAH_DATA_DIR"] = "/content/drive/MyDrive/mtah2027/data"   # PRIMA degli import
import os, sys, time
ROOT = os.environ.get("MTAH_ROOT") or os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if not os.path.exists(os.path.join(ROOT, "games.py")):      # notebook lanciato dalla root
    ROOT = os.getcwd()
sys.path[:0] = [ROOT, os.path.join(ROOT, "solution"), os.path.join(ROOT, "evaluation")]
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore", category=RuntimeWarning)
print("ROOT =", ROOT)

In [ ]:
from possession import pipeline, data
TEAM_NAME = "team_name"        # <- il nome della squadra: il file deve chiamarsi <team_name>.csv
params, cfg = pipeline.load_params()
print("params.json presente:", os.path.exists(pipeline.PARAMS_PATH)); print(cfg)
t = time.time(); res = pipeline.run(); print(f"{time.time()-t:.1f}s")
labels = res["labels"]

In [ ]:
# Struttura dei possessi previsti (si guarda il NOSTRO output, non la ground truth)
for half, a, b in data.graded_blocks():
    s = labels[a:b+1]; edges = np.flatnonzero(s[1:] != s[:-1]) + 1
    segs = np.split(s, edges); pos = [x for x in segs if x[0] in "HA"]
    print(f"tempo {half}: {len(pos)} possessi, durata media {np.mean([len(x) for x in pos])/25:.1f}s, "
          f"pause D: {sum(x[0]=='D' for x in segs)}")

In [ ]:
# Una finestra per vedere punteggi e decisioni
a, b = 30000, 33000; x = np.arange(a, b)/25; ll = res["scores"]
fig, ax = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
ax[0].plot(x, ll[a:b, 0]-ll[a:b, 1], lw=.7, label="evidenza H - A"); ax[0].plot(x, ll[a:b, 2], lw=.7, label="evidenza D"); ax[0].legend()
ax[1].plot(x, [{"H":0,"A":1,"D":2,"X":np.nan}[c] for c in labels[a:b]]); ax[1].set_yticks([0,1,2], ["H","A","D"]); ax[1].set_xlabel("s")
plt.show()

In [ ]:
out = os.path.join(ROOT, "evaluation", "results", f"{TEAM_NAME}.csv")
pipeline.write_submission(labels, out)
import subprocess
print(subprocess.run([sys.executable, os.path.join(ROOT, "evaluation", "validate_submission.py"), out],
                     capture_output=True, text=True).stdout)

## Controllo di correttezza sui 5 minuti pubblici

⚠️ **Solo** per verificare che la pipeline sia corretta (frame allineati, etichette sensate). Il regolamento vieta di usarli per scegliere tra versioni o tarare parametri: 17 possessi non bastano e portano a scelte sbagliate.

In [ ]:
print(subprocess.run([sys.executable, os.path.join(ROOT, "evaluation", "run_evaluation.py")],
                     capture_output=True, text=True, cwd=ROOT).stdout[:3000])

## Riproducibilità: due esecuzioni devono dare lo stesso file

In [ ]:
import hashlib
h1 = hashlib.sha256(open(out, "rb").read()).hexdigest()
res2 = pipeline.run(verbose=False); tmp = out + ".check"; pipeline.write_submission(res2["labels"], tmp)
h2 = hashlib.sha256(open(tmp, "rb").read()).hexdigest(); os.remove(tmp)
print("identico" if h1 == h2 else "DIVERSO!", h1[:16])